In [4]:
!pip install -q taco-box jiwer evaluate
!git clone https://github.com/dll-ncai/Online-Urdu-HWR.git /kaggle/working/repo

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.1 MB/s eta 0:00:00a 0:00:01
Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 11), reused 51 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 444.99 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [5]:
import os
print(os.path.exists("/kaggle/working/repo/model/joint_model.py"))

True


In [7]:
CORE_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core"
GEN_ROOT  = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_generative"
CKPT_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints"
CKPT_DIR  = f"{CKPT_ROOT}/checkpoints_folder/checkpoints_folder/partials"
WORK = "/kaggle/working"

import os, sys
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo")

In [8]:
import subprocess
subprocess.run(f"mkdir -p {WORK}/merged/Dataset", shell=True, check=True)
subprocess.run(f"cp -rs {CORE_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
subprocess.run(f"cp -rs {GEN_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
RAW_ROOT = f"{WORK}/merged"

In [9]:
import pandas as pd
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{CORE_ROOT}/{split}.csv")
    df.to_csv(f"{WORK}/{split}_leakproof.csv", index=False)

In [10]:
import torch
from torch.utils.data import DataLoader
from utils.dataset import ocollate_fn, OHWRDataset
from model.joint_multi_modality_model import JointMultiModel
from tokeniser import get_tokenizer
from utils.losses import JointLoss
import importlib.util

device = torch.device("cuda")

def get_tokenizer_fixed():
    tok = get_tokenizer()
    tok.add_special_tokens({'bos_token': '<s>', 'eos_token': '</s>', 'pad_token': '<pad>',
                             'unk_token': '<unk>', 'mask_token': '<mask>'})
    return tok

tokenizer = get_tokenizer_fixed()

spec2 = importlib.util.spec_from_file_location("onl", "/kaggle/working/repo/train_uhwr_online_camera_ready.py")
onl = importlib.util.module_from_spec(spec2)
spec2.loader.exec_module(onl)

fusion_loss_fn = JointLoss(blank_id=tokenizer.vocab_size, ctc_weight=0.8, ce_weight=0.2)

In [11]:
import zipfile

src_dir = "/kaggle/input/datasets/eshalfatima05/best-fusion/best_fusion"
FUSION_PATH = "/kaggle/working/best_fusion.pt"

with zipfile.ZipFile(FUSION_PATH, "w", zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(src_dir):
        for f in files:
            full = os.path.join(root, f)
            arcname = os.path.join("best_fusion", os.path.relpath(full, src_dir))
            zf.write(full, arcname)

sd = torch.load(FUSION_PATH, map_location="cpu", weights_only=True)
print(len(sd), list(sd.keys())[:5])

718 ['cnn_encoder.cnn_encoder.original_encoder.conv1.0.weight', 'cnn_encoder.cnn_encoder.original_encoder.conv1.0.bias', 'cnn_encoder.cnn_encoder.original_encoder.conv1.1.weight', 'cnn_encoder.cnn_encoder.original_encoder.conv1.1.bias', 'cnn_encoder.cnn_encoder.original_encoder.conv1.1.running_mean']


In [12]:
aux_feat = ["img_dx","img_dy","img_sin_theta","img_cos_theta","img_curvature",
            "img_speed","img_acceleration","img_time_norm","img_pressure",
            "img_x_tilt","img_y_tilt"]

test_df = pd.read_csv(f"{WORK}/test_leakproof.csv")
fusion_test_ds = OHWRDataset(RAW_ROOT, test_df, tokenizer, img_feat="img_stroke", aux_feat=aux_feat)
fusion_test_loader = DataLoader(fusion_test_ds, batch_size=32, shuffle=False, collate_fn=ocollate_fn, num_workers=0)

fusion_model = JointMultiModel(
    trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
    tokenizer=tokenizer,
    trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
    freeze_decoder=True,
    encoder_path=f"{CKPT_DIR}/best_transformer_encoder_uhwr_icdar.pt",
    decoder_path=f"{CKPT_DIR}/best_transformer_decoder_uhwr_icdar.pt",
    cnn_encoder_path=f"{CKPT_DIR}/best_cnn_encoder_uhwr_icdar.pt",
    ctc_head_path=f"{CKPT_DIR}/best_ctc_head_uhwr_icdar.pt",
    img_feat="img_stroke", aux_feat=aux_feat,
    freeze_pcnn_encoder=False, freeze_tr_encoder=True,
).to(device)
fusion_model.load_state_dict(sd)
fusion_model.eval()

test_loss, test_cer = onl.evaluate(fusion_model, fusion_test_loader, tokenizer, device, fusion_loss_fn, decode_mode="beam_search")
print("Fusion TEST CER:", test_cer)

Loaded pretrained original CNN.
Created fused encoder | Fusion: adaptive | Aux branches: 11
Loaded Transformer encoder weights from /kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints/checkpoints_folder/checkpoints_folder/partials/best_transformer_encoder_uhwr_icdar.pt
Loaded Transformer decoder weights from /kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints/checkpoints_folder/checkpoints_folder/partials/best_transformer_decoder_uhwr_icdar.pt
Loaded CTC head weights from /kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints/checkpoints_folder/checkpoints_folder/partials/best_ctc_head_uhwr_icdar.pt
Transformer encoder frozen.


Evaluating: 100%|██████████| 8/8 [11:07<00:00, 83.43s/it]

Fusion TEST CER: 0.044379676153222455
